In [ ]:
from langchain_ollama import ChatOllama
from langchain_community.vectorstores import FAISS
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from pathlib import Path
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions

In [ ]:
input_folder = Path("../DDRM")
output_folder = Path("../DDRM")
output_folder.mkdir(exist_ok=True)

pipeline_options = PdfPipelineOptions(
    do_ocr=False,              
    generate_page_images=False
)

pdf_format = PdfFormatOption(
    pipeline_options=pipeline_options,
    extract_images=False,
    image_description=False
)

converter = DocumentConverter(
    format_options={InputFormat.PDF: pdf_format}
)

all_markdowns = []
for pdf_file in input_folder.glob("*.pdf"):
    print(f"Traitement de : {pdf_file.name}")
    result = converter.convert(pdf_file)
    markdown_text = result.document.export_to_markdown()

    output_md = output_folder / f"{pdf_file.stem}.md"
    output_md.write_text(markdown_text, encoding="utf-8")

    all_markdowns.append(markdown_text)
    print(f"✓ {output_md.name} sauvegardé.")

Traitement de : DDRM_Vienne_2024-1-10.pdf
✓ DDRM_Vienne_2024-1-10.md sauvegardé.
Traitement de : DDRM_Vienne_2024.pdf
✓ DDRM_Vienne_2024.md sauvegardé.


In [ ]:
def chunk_markdown_by_heading(md_text, heading_level="##"):
    chunks = []
    lines = md_text.splitlines()
    current_title = None
    current_content = []

    for line in lines:
        if line.startswith(heading_level):
            if current_title:
                chunks.append({"title": current_title, "content": "\n".join(current_content)})
            current_title = line.replace(heading_level, "").strip()
            current_content = []
        else:
            current_content.append(line)
    if current_title:
        chunks.append({"title": current_title, "content": "\n".join(current_content)})

    return chunks

all_chunks = []
for md in all_markdowns:
    all_chunks.extend(chunk_markdown_by_heading(md))

print(f"Nombre total de chunks créés : {len(all_chunks)}")

Nombre total de chunks créés : 205


In [ ]:
embedder = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

texts_for_embeddings = [f"{chunk['title']} : {chunk['content']}" for chunk in all_chunks]

vectorstore = FAISS.from_texts(texts_for_embeddings, embedding=embedder)
print("Embeddings et vecteur FAISS créés ✅")

SYSTEM_PROMPT = (
    "Tu es un assistant d'analyse documentaire (DDRM / risques). "
    "RÈGLE ABSOLUE : tu dois répondre UNIQUEMENT à partir du CONTEXTE "
    "fourni par l'outil search_ddrm. "
    "Si le contexte ne contient pas l'information, réponds exactement : "
    "'Information non trouvée dans les documents fournis.' "
    "Réponds en français, clair, structuré. "
    "Ajoute des citations au format "
    "[source: <source>, section: <section>]."
)
chat_model = ChatOllama(model="ministral-3", prompt=SYSTEM_PROMPT)

prompt_template = ChatPromptTemplate.from_template(
    "Tu es un assistant expert en risques majeurs. "
    "Réponds aux questions de manière concise en utilisant le contexte suivant:\n\n"
    "{context}\n\nQuestion: {question}\nRéponse:"
)


def ask_question(question, k=5):
    docs = vectorstore.similarity_search(question, k=k)
    context = "\n\n".join([d.page_content for d in docs])
    
    prompt = prompt_template.format(
        context=context,
        question=question
    )

    response = chat_model.invoke(prompt)

    return response.content


question = "Quels sont les risques majeurs en Vienne ?"
answer = ask_question(question)
print("\nQuestion:", question)
print("Réponse:", answer)


Embeddings et vecteur FAISS créés ✅

Question: Quels sont les risques majeurs en Vienne ?
Réponse: Dans le département de la **Vienne**, les risques majeurs sont **peu nombreux**, mais les principaux concernent :

1. **Inondations** (notamment dans les zones proches des cours d’eau comme la Vienne, la Gartempe ou la Creuse).
2. **Mouvements de terrain** (glissements de terrain et affaissements de terrain karstiques, localisés).
3. **Sécheresses prolongées** (impactant l’agriculture et les ressources en eau).

*Aucun risque majeur lié aux séismes, volcans ou tempêtes extrêmes n’est signalé.*
